# Crash!💥 Boom!💥 Bang! 💥🚗🚙💥🚗
**A SQL based exploratory data analysis of traffic accidents in Okinawa, Japan**
  
---   
# <span style="color:#0279f0;">Part 3: Spatial Analysis</span>
---
Now that we have a big picture overview of the accidents in 2024, let's take a closer look at WHERE the accidents occurred.  

**Analysis Tasks**    
1. **Accidents by Municipality** - Municipalities and their accident counts
2. **High Risk Roads and Junctions** - Which particular roads or junctions have higher accident counts?
3. **Black Spot Analysis** - Which areas show the highest concentration of accidents?

**Formula**  
`Accident Share (%) = Municipality Accident Count / Total Accidents × 100`

**Tools used in this notebook:**
- Pandas – for loading the CSV, quick checks and dataframes for basic data manipulation.
- DuckDB – for all aggregation queries
- Matplotlib / Seaborn – for visualisations

##### 👩🏻‍💻  *<span style="color:#0279f0">Loading data to begin...</span>*

In [1]:
import sys
from pathlib import Path


project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from sql.notebook_setup import *

# Load data with pandas
accidents_df = load_data(FILTERED_DATA_PATH)
municipal_df = load_data(MUNICIPAL_DATA_PATH)

# Connect to DuckDB
conn = create_connection()
# Register dataframes with DuckDB
register_table(conn, "accidents", accidents_df)
register_table(conn, "municipality", municipal_df)

#Verify dataset is loaded
print_summary(accidents_df,"`Accidents` Table",10)
print_summary(municipal_df,"`Municipality` Codes Table")



✅ `Accidents` Table: 2,749 records
----------------------------------------
✅ Showing 10 of 68 columns:
 1. data_category
 2. prefecture_code
 3. police_station_code
 4. report_id
 5. accident_details_code
 6. fatalities
 7. injured_persons
 8. route_code
 9. location_code
10. municipality_code
... 58 additional columns

✅ `Municipality` Codes Table: 41 records
----------------------------------------
✅ Showing 3 of 3 columns:
 1. municipality_code
 2. municipality_name
 3. region


..........  

### **3.1 Accident Counts by Municipality & Region**  
How do different municipals and regions fare in terms of accident count?

##### 👩🏻‍💻  *<span style="color:#0279f0">SQL: Analyzing Municipality Accident Counts...</span>*

In [2]:
# ==========================
# 3.1 Accident Counts by Municipality & Region
# Step 1. Create the temporary view
# Step 2. Total accidents per municipality
# Step 3. Aggregate top 4 municipalities with the most accidents
# Step 4. Aggregate by region
# ==========================

# Step 1: Create the temporary view
conn.execute("""
CREATE OR REPLACE TEMP VIEW municipality_counts AS
SELECT
    m.municipality_name,
    COUNT(*) AS accident_count
             
FROM accidents a
LEFT JOIN municipality m
    ON a.municipality_code = m.municipality_code
GROUP BY m.municipality_name;

""")


# Step 2: Aggregate Accident Count by all municipalities
query_municipality ="""
SELECT
  municipality_name,
  accident_count,
  ROUND(accident_count *100 / SUM(accident_count) OVER(), 2) as percentage
FROM municipality_counts
ORDER BY accident_count DESC;
"""


# Print the results
results_municipality = conn.execute(query_municipality).fetchdf()
print("\n----------------------------------------")
print("All Municipalities")
print("----------------------------------------")
display(results_municipality)



----------------------------------------
All Municipalities
----------------------------------------


,municipality_name,accident_count,percentage
0,Naha City,543,19.75
1,Okinawa City,276,10.04
2,Urasoe City,249,9.06
3,Itoman City,233,8.48
4,Uruma City,152,5.53
5,Nago City,140,5.09
6,Haebaru Town,131,4.77
7,Ishigaki City,117,4.26
8,Chatan Town,111,4.04
9,Tomigusuku City,101,3.67


In [3]:
# Step 3: Aggregate Accident Count of Top 4 Municipalities with the most accidents
query_top4 ="""
SELECT
  SUM(accident_count) AS Top_4,
  ROUND(
    SUM(accident_count) *100 / 
    (SELECT SUM(accident_count) FROM municipality_counts),
    2
  ) AS percentage

FROM (
  SELECT accident_count
  FROM municipality_counts
  ORDER BY accident_count DESC
  Limit 4
) top4;
"""


results_top4 = conn.execute(query_top4).fetchdf()
print("\n----------------------------------------")
print("Top 4 Municipalities")
print("----------------------------------------")
display(results_top4)



----------------------------------------
Top 4 Municipalities
----------------------------------------


,Top_4,percentage
0,1301.0,47.33


In [4]:
# Step 4: Aggregate by region

# Pre-check: List of regions and municipalities under each region
# list_region=conn.execute("""SELECT region, municipality_name FROM municipality ORDER BY region, municipality_name;""".fetch_df()
#list_region


# Step 4: Aggregate accident counts by region
query_region ="""
SELECT
  m.region,
  COUNT(*) AS accident_count,
  ROUND(COUNT(*) * 100 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM accidents a
LEFT JOIN municipality m
  ON a.municipality_code = m.municipality_code
GROUP BY m.region
ORDER BY accident_count DESC;

"""


results_region = conn.execute(query_region).fetchdf()
print("\n----------------------------------------")
print("Accident Counts By Region")
print("----------------------------------------")
display(results_region)



----------------------------------------
Accident Counts By Region
----------------------------------------


,region,accident_count,percentage
0,South,1164,42.34
1,Central,1131,41.14
2,North,235,8.55
3,Yaeyama,121,4.40
4,Miyako,98,3.56


### 📊 <span style="color:orange;">3.1 Key Findings: Accidents by Municipality and Region</span>

| Category | Municipality / Region | Accidents | Share | Key Insight |
|---|---|---:|---:|---|
| Top Municipality | Naha City | 543 | 19.75% | Naha City recorded the highest number of accidents, accounting for nearly one in five accidents in Okinawa Prefecture. |
| 2nd Highest | Okinawa City | 276 | 10.04% | Recorded approximately half the number of accidents observed in Naha City. |
| Top 4 Municipalities | Naha City, Okinawa City, Urasoe City, Itoman City | 1,301 | 47.33% | Together, the four municipalities accounted for nearly half of all recorded accidents in the prefecture. |
| South Region | South | 1,164 | 42.34% | Recorded the highest share of accidents, reflecting the concentration of major urban municipalities such as Naha City, Itoman City, and Tomigusuku City. |
| Central Region | Central | 1,131 | 41.14% | Contributed a similarly high share of accidents, driven by municipalities including Okinawa City, Urasoe City, Uruma City, and Ginowan City. |
| South + Central Combined | South & Central | 2,295 | 83.48% | More than four out of every five recorded accidents occurred in the South and Central regions, highlighting the concentration of accidents in Okinawa's primary urban and suburban areas. |
| North Region | North | 235 | 8.55% | Recorded substantially fewer accidents than the South and Central regions, consistent with lower population density and traffic volumes. |
| Yaeyama Region | Yaeyama | 121 | 4.40% | Represented a relatively small share of prefecture-wide accidents, with most occurring in Ishigaki City. |
| Miyako Region | Miyako | 98 | 3.56% | Accounted for the smallest regional share among the inhabited island groups analyzed. |

..........  

### **3.2 High Risk Roads & Junctions**  
Now that we have identified the high risk municipalities, it's time to zoom in and find out which roads and junctions within these areas are the most dangerous.  

##### 👩🏻‍💻  *<span style="color:#0279f0">Analyzing Roads...</span>*

In [5]:
query_routes = """
SELECT
  route_code,
  COUNT(*) as accident_count,
  SUM(fatalities) as fatalities,
  SUM(injured_persons) as injuries,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM accidents
GROUP BY route_code
ORDER BY accident_count DESC
LIMIT 10
"""

result_routes = conn.execute(query_routes).fetchdf()
print("\n\n----------------------------------------")
print("💥Top 10 Routes By Accidents:")
print("----------------------------------------")
display(result_routes)


query_routes_kp = """
SELECT
    route_code,
    location_code AS km_post_raw,
    ROUND(location_code / 10.0, 2) AS km_post,
    COUNT(*) AS accident_count,
    SUM(fatalities) AS fatalities,
    SUM(injured_persons) AS injured
FROM accidents
WHERE location_code <> 0
GROUP BY ALL
ORDER BY accident_count DESC
LIMIT 30
"""

results_routes_kp = conn.execute(query_routes_kp).fetchdf()
print("\n\n----------------------------------------")
print("💥Top 30 Route & KM-Post By Accidents:")
print("----------------------------------------")
display(results_routes_kp)




----------------------------------------
💥Top 10 Routes By Accidents:
----------------------------------------


,route_code,accident_count,fatalities,injuries,percentage
0,580,322,9.0,426.0,11.71
1,32010,196,2.0,217.0,7.13
2,3300,178,1.0,213.0,6.48
3,3290,169,5.0,208.0,6.15
4,99000,145,0.0,169.0,5.27
5,32080,88,0.0,97.0,3.20
6,32110,79,0.0,89.0,2.87
7,3310,70,3.0,77.0,2.55
8,32050,70,0.0,77.0,2.55
9,10820,56,1.0,68.0,2.04




----------------------------------------
💥Top 30 Route & KM-Post By Accidents:
----------------------------------------


,route_code,km_post_raw,km_post,accident_count,fatalities,injured
0,27560,115,11.5,7,0.0,9.0
1,580,1192,119.2,5,0.0,5.0
2,580,1069,106.9,5,0.0,6.0
3,580,1121,112.1,5,0.0,6.0
4,3290,671,67.1,5,0.0,5.0
5,580,1087,108.7,5,0.0,5.0
6,580,672,67.2,5,0.0,5.0
7,3300,57,5.7,5,0.0,7.0
8,580,1100,110.0,5,0.0,5.0
9,580,1171,117.1,4,0.0,7.0


### 📊 <span style="color:orange;">3.2.1 Key Findings: Accidents by Route</span>

#### Top 10 Routes by Accident Count

| Rank | Route Code | Accidents | Share | Observation |
|---:|---:|---:|---:|---|
| 1 | 580 | 322 | 11.71% | National Highway Route 58 |
| 2 | 32010 | 196 | 7.13% | General Municipal Road (Naha) |
| 3 | 3300 | 178 | 6.48% | General Municipal Road (Route 330) |
| 4 | 3290 | 169 | 6.15% | General Municipal Road (Route 329) |
| 5 | 99000 | 145 | 5.27% | Other Public Roads |
| 6 | 32080 | 88 | 3.20% | General Municipal Road (Naha) |
| 7 | 32110 | 79 | 2.87% | General Municipal Road (Naha) |
| 8 | 32050 | 70 | 2.55% | General Municipal Road (Naha) |
| 9 | 3310 | 70 | 2.55% | General Municipal Road (Route 331) |
| 10 | 10820 | 56 | 2.04% | Major Prefectural Road (Route 82) |

#### Key Insights

- **National Highway Route 58 (Route Code 580)** recorded **322 accidents (11.71%)**. It is the highest-risk route in the dataset with an accident count that's substantially higher than any other route.

- **Naha municipal roads** (route codes beginning with **32**) appear **five times** in the top 10. This highlights the concentration of accidents within Okinawa's capital and its surrounding urban road network.

- **Routes 329 and 330** ranked **3rd** and **4th**, indicating that Okinawa's eastern arterial corridor also experiences a high volume of traffic accidents.

- Together, the top-ranked routes indicate that accidents are concentrated on Okinawa's primary urban and arterial road network rather than being evenly distributed across the prefecture.


### 📍 <span style="color:orange;">3.2.2 Key Findings: Route & KM-Post Analysis</span>

#### Key Insights

- **Route 58** dominated the hotspot analysis, appearing **18 times** in the top 30 Route & KM-Post combinations. Most accident-prone sections were concentrated between **KP 54.6 and KP 119.2**, spanning approximately **Chatan to Nago**.

- The most severe sections on **Route 58** were located at **KP 109.2** (1 fatality) and **KP 115.9** (2 fatalities), both within **Nago City**.

- **Route 329** showed a cluster of accidents between **KP 48.2 and KP 67.1**, suggesting a potentially high-risk section along the central-eastern corridor.

- **Route 330** accidents were concentrated within the first **15 km** (KP 4.0–13.8). This indicates that the southern section of the route is particularly accident-prone.

- **Prefectural Route 256 (Route Code 27560)** ranked **first** in the KM-Post analysis with **7 accidents at KP 11.5**, despite not appearing among the top 10 routes overall. This suggests that while the route has relatively few accidents overall, a specific location represents a localized accident hotspot.

#### Next Step

While the Route & KM-Post analysis identifies high-risk road sections, a dedicated spatial visualization using **Tableau** will provide a clearer representation of accident hotspots by mapping latitude and longitude coordinates. This spatial analysis is presented in **Section 3.3: Geographic Hotspot Analysis**.

---  

### **3.3 Geographic HotSpot Analysis**  
Are there any areas where accidents repeatedly occur? Analysing latitude and longitude data will help us identify such areas.  
  
Accidents that occur within an approximate radius of 110m of each other will be grouped as a cluster. Clusters with the highest accident counts will be identified as black spots.

#### <span style="color:orange;">Note on limitations of SQL only approach:</span>  

This analysis uses a simplified coordinate rounding method to identify black spots. As this project is primarily focused on SQL-based analysis, advanced spatial clustering algorithms like DBSCAN scikit-learn are outside its scope. The rounding method should provide us with a sufficiently accurate approximation for identifying high-risk locations at the intersection level.  

#### 👩🏻‍💻 *<span style="color:#0279f0;">SQL: Identifying Geographic Hotspots...*</span>  

In [6]:
#Accidents that fall within an approximate radius of 110 meters of each other.
#extract degrees, minutes and seconds from the raw value,
#convert each to decimal degrees, add them together,


query_blackspots = """

WITH coordinates AS (
    
    SELECT
        ROUND(
            (latitude_north / 10000000) +
            ((latitude_north % 10000000) / 100000 / 60.0) +
            ((latitude_north % 100000) / 1000.0 / 3600.0),
            3
        ) AS latitude,

        ROUND(
            (longitude_east / 10000000) +
            ((longitude_east % 10000000) / 100000 / 60.0) +
            ((longitude_east % 100000) / 1000.0 / 3600.0),
            3
        ) AS longitude,

        fatalities,
        injured_persons

    FROM accidents
)

SELECT
    latitude,
    longitude,
    COUNT(*) AS accident_count,
    SUM(injured_persons) AS total_injured,
    SUM(fatalities) AS total_fatalities
FROM coordinates
GROUP BY ALL
ORDER BY accident_count DESC
Limit 30;

"""

blackspots = conn.execute(query_blackspots).fetchdf()
print("\n\n----------------------------------------")
print("💥 Geographic Hotspots:")
print("----------------------------------------")
display(blackspots)




----------------------------------------
💥 Geographic Hotspots:
----------------------------------------


,latitude,longitude,accident_count,total_injured,total_fatalities
0,26.285,128.070,6,6.0,0.0
1,26.350,128.094,6,6.0,0.0
2,26.280,128.168,6,6.0,0.0
3,26.450,128.250,5,6.0,0.0
4,26.512,128.334,4,4.0,0.0
5,26.436,128.132,4,7.0,0.0
6,26.329,128.154,4,4.0,0.0
7,26.976,128.574,4,5.0,0.0
8,26.942,128.578,4,10.0,0.0
9,26.459,128.211,4,4.0,0.0


### 📍 <span style="color:orange;">3.3 Key Findings: Black Spot Analysis</span>

#### Key Findings

- All **top 10 black spots** were located at or immediately next to **road intersections**.
- **No fatalities** were recorded at any of the top 10 black spots, although several locations recorded multiple injuries.
- Most black spots were concentrated in the **southern urban areas** of Okinawa, particularly around **Naha, Ginowan, Tomigusuku, and Chatan**.
- The findings were cross-checked against official **National Police Agency (NPA)** road safety reports. **Isa Intersection (Ginowan)** and **Nakachi East Intersection (Tomigusuku)** were independently identified as major accident locations in 2024.
- Black spots were identified by clustering accidents using rounded coordinates (approximately **110 m**). Large intersections may therefore be split into two nearby clusters.

---

#### Top 10 Black Spots

| Rank | Location | Accidents | Injured | Fatalities |
|---:|---|---:|---:|---:|
| 1 | Route 58 / Route 81, Isa (Ginowan) | 6 | 8 | 0 |
| 2 | Kumoji, Naha City | 6 | 8 | 0 |
| 3 | Route 58 / Shisa Street | 6 | 9 | 0 |
| 4 | Route 82 / Route 29 (Naha–Itoman Line) | 5 | 8 | 0 |
| 5 | Tomigusuku Nakachi IC | 5 | 7 | 0 |
| 6 | Route 58 / Route 130 | 5 | 6 | 0 |
| 7 | Route 48 / Route 507 | 5 | 6 | 0 |
| 8 | Route 329 / Route 26 / Kariyushi Dori | 5 | 5 | 0 |
| 9 | Chatan Town (off Route 58) | 4 | 4 | 0 |
| 10 | Route 82 / Route 329, Naha City | 4 | 5 | 0 |


#### 🔎 Observation

**Tomigusuku Nakachi IC** may be the largest accident hotspot in the dataset. The intersection appears as **two adjacent coordinate clusters** (26.176, 127.668 and 26.176, 127.669) because of coordinate rounding. When considered together, these two clusters likely represent the highest concentration of accidents in the dataset.


#### Possible Contributing Factors

The data does not identify the causes of these accidents. However, possible contributing factors include:

- High traffic volume
- Multiple turning and merging movements
- Complex intersection layouts
- Traffic signal timing

Further investigation is recommended to determine whether improvements such as signal timing changes, road design changes, or targeted enforcement could reduce accidents at these locations.